In [2]:
# gdf_all.to_parquet("data/osm/all_highways.parquet", index=False)    

In [ ]:
import sys
from pathlib import Path

# Notebook liegt in preprocessing/; pipeline.py daneben.
sys.path.insert(0, str(Path.cwd()))

import geopandas as gpd
import pandas as pd

from pipeline import (
    run_pipeline,
    load_bkg_layers,
    discover_pbfs,
    process_region,
    _concat_slim,
    attach_admin,
    join_coverage,
    normalize_highway,
    aggregate_level,
    export_pmtiles,
    EXPORT_SPECS,
    EBENEN,
    COVERAGE_CSV_URL,
)

DATA = Path("data")
BKG_GPKG = DATA / "bkg" / "DE_VG5000.gpkg"

In [ ]:
# Schritt-für-Schritt für interaktive Exploration.
# Für headless-Runs (z.B. Docker) genügt:
#     summary = run_pipeline(DATA, DATA, limit_regions=["DE-HB", "DE-HH"], dry_run=True)

# OSM-PBF-Verzeichnis kann separat gesetzt werden (Server: woanders als das Repo).
OSM_DIR = DATA / "osm"   # Server: Path("/home/simon/mapillary_coverage/data/osm/processed")

# 1) BKG-Verwaltungsebenen einmalig laden (residiert während des Loops)
gem_hierarchy, gdf_lan, gdf_krs, gdf_gem = load_bkg_layers(BKG_GPKG)

# 2) PBFs pro Region einlesen — load_pbf_filtered() liest nur Major Roads,
#    spart ~90% RAM gegenüber dem alten read_file(layer="lines").
pbfs = discover_pbfs(OSM_DIR)
print(f"{len(pbfs)} PBFs gefunden")

slim_frames = [process_region(p) for p in pbfs]
merged_gdf_clean_majorRoads = _concat_slim(slim_frames)
del slim_frames

print(f"Major Roads gesamt: {len(merged_gdf_clean_majorRoads):,}")

In [ ]:
# Coverage-CSV wird direkt vom Repo gezogen (immer aktuell):
coverage_data = pd.read_csv(COVERAGE_CSV_URL)

In [7]:
coverage_data.head()

,osm_id,mapillary_coverage
0,1451397819,pano
1,985424701,pano
2,985157297,pano
3,984815229,pano
4,984730457,pano


In [ ]:
# 3) Admin-Join (Bundesland/Kreis/Gemeinde/AGS_0) + Coverage + length_m + highway normalisieren
lines = attach_admin(merged_gdf_clean_majorRoads, gem_hierarchy)
lines = join_coverage(lines)  # default = COVERAGE_CSV_URL
lines["length_m"] = lines.geometry.length
lines = normalize_highway(lines)

# Aliase für die unten folgenden Anzeige-Zellen (alte Variablennamen):
merged_gdf = lines
df = lines

In [10]:
merged_gdf

,osm_id,name,highway,waterway,aerialway,barrier,man_made,railway,z_order,other_tags,geometry,region,mapillary_coverage
0,3996955,None,motorway,None,None,None,None,None,9,"""check_date:lit""=>""2021-04-06"",""embankment""=>""...","LINESTRING (13.09264 52.31368, 13.09376 52.315...",DE-BB,regular
1,3996957,None,motorway,None,None,None,None,None,9,"""destination:arrow:lanes""=>""through|through|th...","LINESTRING (13.09522 52.3023, 13.09285 52.30184)",DE-BB,pano
2,4040461,None,motorway,None,None,None,None,None,9,"""int_ref""=>""E 26"",""lanes""=>""2"",""lit""=>""no"",""ma...","LINESTRING (11.92993 53.3017, 11.92475 53.3032...",DE-BB,pano
3,4040465,None,motorway,None,None,None,None,None,9,"""int_ref""=>""E 26"",""lanes""=>""2"",""lit""=>""no"",""ma...","LINESTRING (12.06881 53.28689, 12.06785 53.287...",DE-BB,regular
4,4040467,None,motorway,None,None,None,None,None,9,"""int_ref""=>""E 26"",""lanes""=>""2"",""lit""=>""no"",""ma...","LINESTRING (12.13879 53.26535, 12.13794 53.265...",DE-BB,regular
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16546271,1475095963,None,steps,None,None,None,None,None,0,"""access""=>""private""","LINESTRING (10.73222 50.82282, 10.73231 50.82281)",DE-TH,NaN
16546272,1475095964,None,footway,None,None,None,None,None,0,"""access""=>""private""","LINESTRING (10.73243 50.82301, 10.73267 50.82296)",DE-TH,NaN
16546273,1475095965,None,steps,None,None,None,None,None,0,"""access""=>""private""","LINESTRING (10.73287 50.82273, 10.7328 50.82274)",DE-TH,NaN
16546274,1475095966,None,steps,None,None,None,None,None,0,"""access""=>""private""","LINESTRING (10.73285 50.82268, 10.73278 50.82269)",DE-TH,NaN


In [12]:
#merged_gdf_clean.highway.unique()

In [14]:
merged_gdf_clean_majorRoads

,osm_id,highway,region,mapillary_coverage,geometry
0,3996955,motorway,DE-BB,regular,"LINESTRING (13.09264 52.31368, 13.09376 52.315..."
1,3996957,motorway,DE-BB,pano,"LINESTRING (13.09522 52.3023, 13.09285 52.30184)"
2,4040461,motorway,DE-BB,pano,"LINESTRING (11.92993 53.3017, 11.92475 53.3032..."
3,4040465,motorway,DE-BB,regular,"LINESTRING (12.06881 53.28689, 12.06785 53.287..."
4,4040467,motorway,DE-BB,regular,"LINESTRING (12.13879 53.26535, 12.13794 53.265..."
...,...,...,...,...,...
16546196,1474827547,tertiary,DE-TH,NaN,"LINESTRING (12.57312 50.92311, 12.57347 50.9231)"
16546219,1474895413,tertiary,DE-TH,NaN,"LINESTRING (9.86663 50.68385, 9.86676 50.6839,..."
16546241,1475048515,tertiary,DE-TH,NaN,"LINESTRING (12.46595 50.94996, 12.46601 50.94997)"
16546242,1475048516,tertiary,DE-TH,NaN,"LINESTRING (12.47406 50.97261, 12.47432 50.972..."


### BKG als einzige Quelle (3 Ebenen)

- **Bundesland** (lan), **Kreis** (krs), **Gemeinde** (gem) kommen nur noch aus dem VG5000-GPKG unter `preprocessing/data/bkg/`.
- Ein räumlicher Join erfolgt einmal gegen die View `v_vz5000_gem` (enthält GEN_L, GEN_K, GEN_G) → jede Straße bekommt alle drei Zuordnungen.
- Spaltenname im GDF: **Kreis** (BKG spricht von "Kreis", nicht "Landkreis"); Aggregation über `r_einheit` wählbar.

In [47]:
gdf_gem

,OBJID,BEGINN,ADE,GF,BSG,LKZ,ARS,AGS,SDV_ARS,Gemeinde,...,SN_K,SN_V1,SN_V2,SN_G,FK_S3,NUTS,ARS_0,AGS_0,WSK,geometry
0,DEBKGVG5000008NT,2019-10-04,6,9,1,SH,010010000000,01001000,010010000000,Flensburg,...,01,00,00,000,R,DEF01,010010000000,01001000,2008-01-01,"MULTIPOLYGON (((531633.213 6075130.547, 532385..."
1,DEBKGVG5000008NU,2019-10-04,6,9,1,SH,010020000000,01002000,010020000000,Kiel,...,02,00,00,000,R,DEF02,010020000000,01002000,2006-01-01,"MULTIPOLYGON (((575109.999 6031870.053, 575829..."
2,DEBKGVG5000008NV,2019-10-04,6,9,1,SH,010030000000,01003000,010030000000,Lübeck,...,03,00,00,000,R,DEF03,010030000000,01003000,2006-02-01,"MULTIPOLYGON (((621250.836 5983463.205, 620859..."
3,DEBKGVG5000008NW,2019-10-04,6,9,1,SH,010040000000,01004000,010040000000,Neumünster,...,04,00,00,000,R,DEF04,010040000000,01004000,1970-04-26,"MULTIPOLYGON (((564022.128 6000401.822, 564959..."
4,DEBKGVG5000008NX,2019-10-04,6,9,1,SH,010510011011,01051011,010510011011,Brunsbüttel,...,51,00,11,011,R,DEF05,010510011011,01051011,2009-01-01,"MULTIPOLYGON (((507163.312 5976634.451, 507870..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10944,DEBKGVG5000008NQ,2019-10-04,6,9,1,TH,160775051011,16077011,160775051011,Göpfersdorf,...,77,50,51,011,R,DEG0M,160775051011,16077011,2018-07-06,"MULTIPOLYGON (((751999.518 5645268.015, 752651..."
10945,DEBKGVG5000008NR,2019-10-04,6,9,1,TH,160775051023,16077023,160775051023,Langenleuba-Niederhain,...,77,50,51,023,R,DEG0M,160775051023,16077023,2018-07-06,"MULTIPOLYGON (((745945.807 5655517.064, 747488..."
10946,DEBKGVG5000008NS,2019-10-04,6,9,1,TH,160775051036,16077036,160775051036,Nobitz,...,77,50,51,036,R,DEG0M,160775051036,16077036,2018-07-06,"MULTIPOLYGON (((743071.244 5654791.729, 744601..."
10947,DEBKGVG5000008MZ,2020-11-08,6,9,1,TH,160775052003,16077003,160775052003,Dobitschen,...,77,50,52,003,R,DEG0M,160775052003,16077003,2019-01-01,"MULTIPOLYGON (((728904.255 5650186.152, 730841..."


In [48]:
# Optional: Stichprobe der BKG-Daten
# gem_hierarchy[["Bundesland","Kreis","Gemeinde"]].head()
# gdf_lan[["Bundesland"]].head()

In [49]:
# Spalten sind bereits in load-Zelle benannt (Bundesland, Kreis, Gemeinde)
# Kein Rename nötig bei BKG.


In [55]:
lines

,osm_id,highway,region,mapillary_coverage,geometry,Bundesland,Kreis,Gemeinde,AGS_0,length_m
0,3996955,motorway,DE-BB,regular,"LINESTRING (778933.815 5803816.85, 778999.282 ...",Brandenburg,Potsdam-Mittelmark,Michendorf,12069397,489.512134
1,3996957,motorway,DE-BB,pano,"LINESTRING (779181.582 5802561.592, 779023.187...",Brandenburg,Potsdam-Mittelmark,Nuthetal,12069454,169.464392
2,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Mecklenburg-Vorpommern,Ludwigslust-Parchim,Ruhner Berge,13076168,2089.505867
3,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Mecklenburg-Vorpommern,Ludwigslust-Parchim,Ruhner Berge,13076168,2089.505867
4,4040465,motorway,DE-BB,regular,"LINESTRING (704553.042 5908579.118, 704487.691...",Brandenburg,Prignitz,Putlitz,12070325,5345.608668
...,...,...,...,...,...,...,...,...,...,...
1606488,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Hessen,Fulda,Rasdorf,06631022,614.275539
1606489,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Hessen,Fulda,Rasdorf,06631022,614.275539
1606490,1475048515,tertiary,DE-TH,NaN,"LINESTRING (743434.92 5649981.946, 743439.38 5...",Thüringen,Altenburger Land,Altenburg,16077001,4.590947
1606491,1475048516,tertiary,DE-TH,NaN,"LINESTRING (743885.838 5652526.263, 743904.698...",Thüringen,Altenburger Land,Nobitz,16077036,151.444109


In [57]:
df

,osm_id,highway,region,mapillary_coverage,geometry,Bundesland,Kreis,Gemeinde,AGS_0,length_m
0,3996955,motorway,DE-BB,regular,"LINESTRING (778933.815 5803816.85, 778999.282 ...",Brandenburg,Potsdam-Mittelmark,Michendorf,12069397,489.512134
1,3996957,motorway,DE-BB,pano,"LINESTRING (779181.582 5802561.592, 779023.187...",Brandenburg,Potsdam-Mittelmark,Nuthetal,12069454,169.464392
2,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Mecklenburg-Vorpommern,Ludwigslust-Parchim,Ruhner Berge,13076168,2089.505867
3,4040461,motorway,DE-BB,pano,"LINESTRING (695230.778 5909836.658, 694878.545...",Mecklenburg-Vorpommern,Ludwigslust-Parchim,Ruhner Berge,13076168,2089.505867
4,4040465,motorway,DE-BB,regular,"LINESTRING (704553.042 5908579.118, 704487.691...",Brandenburg,Prignitz,Putlitz,12070325,5345.608668
...,...,...,...,...,...,...,...,...,...,...
1606488,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Hessen,Fulda,Rasdorf,06631022,614.275539
1606489,1474895413,tertiary,DE-TH,NaN,"LINESTRING (561223.013 5615027.469, 561232.308...",Hessen,Fulda,Rasdorf,06631022,614.275539
1606490,1475048515,tertiary,DE-TH,NaN,"LINESTRING (743434.92 5649981.946, 743439.38 5...",Thüringen,Altenburger Land,Altenburg,16077001,4.590947
1606491,1475048516,tertiary,DE-TH,NaN,"LINESTRING (743885.838 5652526.263, 743904.698...",Thüringen,Altenburger Land,Nobitz,16077036,151.444109


In [60]:
#df[df.Gemeinde=="Stetten"].plot()

In [61]:
##### Aggregation: gleiche Pipeline pro Ebene (Bundesland, Kreis, Gemeinde) im Loop

In [ ]:
# Aggregate per Verwaltungs-Ebene
wide_by_ebene = {level: aggregate_level(df, level, gdf_gem) for level in EBENEN}

wide_bundesland = wide_by_ebene["Bundesland"]
wide_kreis      = wide_by_ebene["Kreis"]
wide_gemeinde   = wide_by_ebene["Gemeinde"]
wide = wide_bundesland  # Default für Anzeige

In [63]:
# Stichprobe: Wide-Tabelle Bundesland
wide_by_ebene["Bundesland"].head()


col,Bundesland,all_length_no_cover,all_length_pano,all_length_regular,all_share_no_cover,all_share_pano,all_share_regular,motorway_length_no_cover,motorway_length_pano,motorway_length_regular,...,tertiary_length_regular,tertiary_share_no_cover,tertiary_share_pano,tertiary_share_regular,trunk_length_no_cover,trunk_length_pano,trunk_length_regular,trunk_share_no_cover,trunk_share_pano,trunk_share_regular
0,Baden-Württemberg,18614.1,2702.0,13138.3,0.540253,0.078423,0.381324,147.2,193.8,2309.2,...,3015.2,0.712818,0.075570,0.211612,318.8,20.5,1214.9,0.205119,0.013212,0.781669
1,Bayern,35380.0,3787.0,15811.5,0.643524,0.068881,0.287595,498.5,1538.8,4373.3,...,3201.4,0.834483,0.031105,0.134413,579.0,128.5,955.8,0.348094,0.077271,0.574635
2,Berlin,1218.2,4565.8,2860.4,0.140920,0.528182,0.330899,161.9,399.0,342.5,...,676.4,0.202585,0.515289,0.282126,14.8,1.6,30.3,0.317660,0.033684,0.648656
3,Brandenburg,5124.6,3586.4,8355.4,0.300276,0.210143,0.489581,219.5,548.5,1355.1,...,1911.4,0.422660,0.202258,0.375082,96.5,73.3,252.7,0.228367,0.173518,0.598115
4,Bremen,1464.1,252.5,1386.7,0.471780,0.081372,0.446848,137.8,145.5,430.5,...,289.7,0.672400,0.003758,0.323841,70.7,50.6,159.1,0.252180,0.180400,0.567420


In [64]:
wide_by_ebene["Kreis"].head()


col,Kreis,all_length_no_cover,all_length_pano,all_length_regular,all_share_no_cover,all_share_pano,all_share_regular,motorway_length_no_cover,motorway_length_pano,motorway_length_regular,...,tertiary_length_regular,tertiary_share_no_cover,tertiary_share_pano,tertiary_share_regular,trunk_length_no_cover,trunk_length_pano,trunk_length_regular,trunk_share_no_cover,trunk_share_pano,trunk_share_regular
0,Ahrweiler,391.6,0.0,509.4,0.434605,0.000000,0.565395,10.3,0.0,87.2,...,136.2,0.579610,0.000000,0.420390,15.5,0.0,10.0,0.609346,0.000000,0.390654
1,Aichach-Friedberg,361.9,19.0,133.8,0.703102,0.036878,0.260020,2.5,18.9,20.5,...,14.4,0.938084,0.000000,0.061916,23.1,0.0,0.7,0.969074,0.000000,0.030926
2,Alb-Donau-Kreis,538.3,68.2,560.5,0.461287,0.058424,0.480289,3.7,8.8,104.4,...,162.4,0.628589,0.046323,0.325089,1.4,0.0,14.4,0.087006,0.000000,0.912994
3,Altenburger Land,619.0,42.5,126.9,0.785105,0.053884,0.161011,1.5,0.0,30.0,...,33.0,0.921284,0.002322,0.076394,3.2,11.0,15.6,0.106253,0.368648,0.525100
4,Altenkirchen (Westerwald),610.2,0.0,172.1,0.780025,0.000000,0.219975,0.0,0.0,2.0,...,78.8,0.815468,0.000000,0.184532,3.3,0.0,3.8,0.464311,0.000000,0.535689


In [65]:
# (in Loop oben erledigt; wide_bundesland, wide_kreis, wide_gemeinde verfügbar)

In [66]:
##'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

## create pmtiles

In [67]:
# def fgb_to_pmtiles_layer(polys_fgb, output_pmtiles, layer_name):
#     import subprocess
#     from pathlib import Path

#     subprocess.run([
#         "tippecanoe", "-o", str(Path(output_pmtiles).resolve()),
#         f"--layer={layer_name}",
#         "--minimum-zoom=5", "--maximum-zoom=7",
#         "--force",
#         "--no-feature-limit", "--no-tile-size-limit", 
#         "--drop-densest-as-needed",  # Nur Punkte entfernen, keine Vereinfachung
#         str(Path(polys_fgb).resolve())
#     ], check=True)

#     print("✅ Combined PMTiles created")

In [86]:
##### KREISE (BKG vg5000_krs)

In [ ]:
spec = EXPORT_SPECS["Kreis"]
export_pmtiles(
    wide_kreis,
    gdf_krs[["Kreis", "geometry"]],
    key="Kreis",
    out_fgb=DATA / f"{spec['filename']}.fgb",
    out_pmtiles=DATA / f"{spec['filename']}.pmtiles",
    minzoom=spec["minzoom"],
    maxzoom=spec["maxzoom"],
)

In [88]:
##### GEMEINDEN (BKG vg5000_gem) – viele Features, ggf. höherer minzoom

In [ ]:
spec = EXPORT_SPECS["Gemeinde"]
export_pmtiles(
    wide_gemeinde,
    gdf_gem[["AGS_0", "Gemeinde", "geometry"]],
    key="AGS_0",
    out_fgb=DATA / f"{spec['filename']}.fgb",
    out_pmtiles=DATA / f"{spec['filename']}.pmtiles",
    minzoom=spec["minzoom"],
    maxzoom=spec["maxzoom"],
)

In [90]:
##### BUNDESLÄNDER (BKG vg5000_lan)

In [91]:
# def fgb_to_pmtiles_layer(polys_fgb, output_pmtiles, layer_name):
#     import subprocess
#     from pathlib import Path

#     subprocess.run([
#         "tippecanoe", "-o", str(Path(output_pmtiles).resolve()),
#         f"--layer={layer_name}",
#         "--minimum-zoom=5", "--maximum-zoom=7",
#         "--force",
#         "--no-feature-limit", "--no-tile-size-limit", 
#         "--drop-densest-as-needed",  # Nur Punkte entfernen, keine Vereinfachung
#         str(Path(polys_fgb).resolve())
#     ], check=True)

#     print("✅ Combined PMTiles created")

In [ ]:
spec = EXPORT_SPECS["Bundesland"]
export_pmtiles(
    wide_bundesland,
    gdf_lan[["Bundesland", "geometry"]],
    key="Bundesland",
    out_fgb=DATA / f"{spec['filename']}.fgb",
    out_pmtiles=DATA / f"{spec['filename']}.pmtiles",
    minzoom=spec["minzoom"],
    maxzoom=spec["maxzoom"],
)